# Model Training - INX Future Inc Employee Performance Prediction

## Objective
Build and compare multiple machine learning models to predict employee performance:
1. Logistic Regression
2. Random Forest Classifier
3. Gradient Boosting (XGBoost)
4. Support Vector Machine (SVM)
5. Neural Network (if needed)

## Goal
Select the best model for predicting employee performance to support hiring decisions.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# ML Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

# Handle imbalanced data
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Model persistence
import joblib

import warnings
warnings.filterwarnings('ignore')

# Random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries loaded successfully!")

## 1. Load and Prepare Data

In [ ]:
# Load processed data
df = pd.read_csv('../../data/processed/employee_data_processed.csv')

print(f"Dataset Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")

In [ ]:
# Identify target variable
target_col = [col for col in df.columns if 'performance' in col.lower() and 'rating' in col.lower()]
if target_col:
    target = target_col[0]
else:
    target = 'PerformanceRating'

print(f"Target Variable: {target}")
print(f"\nTarget Distribution:")
print(df[target].value_counts().sort_index())

# Prepare features and target
X = df.drop(columns=[target, 'EmpNumber'] if 'EmpNumber' in df.columns else [target])
y = df[target]

print(f"\nFeature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")

In [ ]:
# Identify feature types
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"\nNumerical Features ({len(numerical_features)}): {numerical_features}")
print(f"\nCategorical Features ({len(categorical_features)}): {categorical_features}")

## 2. Feature Selection and Engineering

In [ ]:
# Load correlation analysis
try:
    correlations = pd.read_csv('../../data/processed/feature_correlations.csv', index_col=0)
    correlations = correlations.iloc[:, 0]  # Get the series
    
    # Select top features based on correlation
    top_numerical = correlations.abs().sort_values(ascending=False).head(15).index.tolist()
    top_numerical = [col for col in top_numerical if col in numerical_features]
    
    print("Top features selected based on correlation analysis:")
    for i, feat in enumerate(top_numerical, 1):
        print(f"  {i}. {feat}: {correlations[feat]:.3f}")
    
    # Update numerical features to use only top features
    numerical_features = top_numerical
except:
    print("Using all numerical features (correlation file not found)")

In [ ]:
# Create preprocessing pipeline
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

# Combine transformers
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])

print("Preprocessing pipeline created successfully!")

## 3. Train-Test Split

In [ ]:
# Split data - 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nTraining set distribution:")
print(y_train.value_counts().sort_index())
print(f"\nTest set distribution:")
print(y_test.value_counts().sort_index())

## 4. Model Training and Evaluation

### 4.1 Define Models

In [ ]:
# Define models to compare
models = {
    'Logistic Regression': LogisticRegression(random_state=RANDOM_STATE, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=RANDOM_STATE, max_depth=10),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE),
    'XGBoost': XGBClassifier(n_estimators=100, random_state=RANDOM_STATE, use_label_encoder=False, eval_metric='mlogloss'),
    'SVM': SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE)
}

print(f"Models to evaluate: {list(models.keys())}")

### 4.2 Train and Evaluate All Models

In [ ]:
# Store results
results = {}
trained_models = {}

print("\n" + "="*100)
print(" " * 40 + "MODEL TRAINING RESULTS")
print("="*100)

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Create pipeline with preprocessing
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    # Train model
    pipeline.fit(X_train, y_train)
    
    # Make predictions
    y_pred = pipeline.predict(X_test)
    y_pred_proba = pipeline.predict_proba(X_test) if hasattr(pipeline, 'predict_proba') else None
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    # Cross-validation score
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='accuracy')
    cv_mean = cv_scores.mean()
    cv_std = cv_scores.std()
    
    # Store results
    results[name] = {
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'CV_Mean': cv_mean,
        'CV_Std': cv_std,
        'predictions': y_pred,
        'probabilities': y_pred_proba
    }
    
    trained_models[name] = pipeline
    
    # Print results
    print(f"  ✓ Accuracy: {accuracy:.4f}")
    print(f"  ✓ Precision: {precision:.4f}")
    print(f"  ✓ Recall: {recall:.4f}")
    print(f"  ✓ F1-Score: {f1:.4f}")
    print(f"  ✓ CV Score: {cv_mean:.4f} (+/- {cv_std:.4f})")

print("\n" + "="*100)
print("Model training completed!")
print("="*100)

### 4.3 Compare Model Performance

In [ ]:
# Create comparison DataFrame
results_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [results[m]['Accuracy'] for m in results.keys()],
    'Precision': [results[m]['Precision'] for m in results.keys()],
    'Recall': [results[m]['Recall'] for m in results.keys()],
    'F1-Score': [results[m]['F1-Score'] for m in results.keys()],
    'CV_Mean': [results[m]['CV_Mean'] for m in results.keys()],
    'CV_Std': [results[m]['CV_Std'] for m in results.keys()]
}).round(4)

results_df = results_df.sort_values('Accuracy', ascending=False)

print("\n" + "="*120)
print(" " * 45 + "MODEL COMPARISON")
print("="*120)
display(results_df)

# Identify best model
best_model_name = results_df.iloc[0]['Model']
best_accuracy = results_df.iloc[0]['Accuracy']

print(f"\n🏆 Best Model: {best_model_name} (Accuracy: {best_accuracy:.4f})")

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']

for idx, (metric, color) in enumerate(zip(metrics, colors)):
    ax = axes[idx // 2, idx % 2]
    
    sorted_data = results_df.sort_values(metric, ascending=True)
    ax.barh(sorted_data['Model'], sorted_data[metric], color=color, edgecolor='black', alpha=0.7)
    ax.set_xlabel(metric, fontsize=12, fontweight='bold')
    ax.set_title(f'Model Comparison - {metric}', fontsize=14, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    
    # Add value labels
    for i, v in enumerate(sorted_data[metric]):
        ax.text(v + 0.005, i, f'{v:.3f}', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../../data/processed/model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Detailed Analysis of Best Model

In [ ]:
# Get best model
best_model = trained_models[best_model_name]
best_predictions = results[best_model_name]['predictions']

# Confusion Matrix
cm = confusion_matrix(y_test, best_predictions)

print(f"\n{best_model_name} - Detailed Performance Analysis")
print("="*80)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, best_predictions))

In [ ]:
# Visualize Confusion Matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
            xticklabels=sorted(y.unique()), yticklabels=sorted(y.unique()))
plt.title(f'Confusion Matrix - {best_model_name}', fontsize=16, fontweight='bold')
plt.ylabel('Actual Performance Rating', fontsize=12)
plt.xlabel('Predicted Performance Rating', fontsize=12)
plt.tight_layout()
plt.savefig('../../data/processed/confusion_matrix_best_model.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Feature Importance Analysis

In [ ]:
# Get feature importance if available
if hasattr(best_model.named_steps['classifier'], 'feature_importances_'):
    # Get feature names after preprocessing
    feature_names = (numerical_features + 
                    list(best_model.named_steps['preprocessor']
                         .named_transformers_['cat']
                         .named_steps['onehot']
                         .get_feature_names_out(categorical_features)))
    
    importances = best_model.named_steps['classifier'].feature_importances_
    
    # Create DataFrame
    feature_importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances
    }).sort_values('Importance', ascending=False)
    
    print("\nTop 15 Most Important Features:")
    print("="*80)
    display(feature_importance_df.head(15))
    
    # Visualize
    plt.figure(figsize=(12, 8))
    top_features = feature_importance_df.head(15)
    plt.barh(range(len(top_features)), top_features['Importance'], color='teal', edgecolor='black')
    plt.yticks(range(len(top_features)), top_features['Feature'])
    plt.xlabel('Feature Importance', fontsize=12, fontweight='bold')
    plt.title(f'Top 15 Feature Importances - {best_model_name}', fontsize=14, fontweight='bold')
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig('../../data/processed/feature_importance.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Save to CSV
    feature_importance_df.to_csv('../../data/processed/feature_importance.csv', index=False)
else:
    print(f"\nFeature importance not available for {best_model_name}")

## 7. Hyperparameter Tuning (Best Model)

In [ ]:
# Hyperparameter tuning for best model
print(f"\nPerforming hyperparameter tuning for {best_model_name}...")

# Define parameter grids for different models
param_grids = {
    'Random Forest': {
        'classifier__n_estimators': [100, 200, 300],
        'classifier__max_depth': [10, 20, 30],
        'classifier__min_samples_split': [2, 5, 10]
    },
    'XGBoost': {
        'classifier__n_estimators': [100, 200, 300],
        'classifier__max_depth': [3, 5, 7],
        'classifier__learning_rate': [0.01, 0.1, 0.3]
    },
    'Gradient Boosting': {
        'classifier__n_estimators': [100, 200, 300],
        'classifier__max_depth': [3, 5, 7],
        'classifier__learning_rate': [0.01, 0.1, 0.3]
    },
    'SVM': {
        'classifier__C': [0.1, 1, 10],
        'classifier__gamma': ['scale', 'auto']
    },
    'Logistic Regression': {
        'classifier__C': [0.1, 1, 10],
        'classifier__penalty': ['l2']
    }
}

if best_model_name in param_grids:
    param_grid = param_grids[best_model_name]
    
    # Grid Search
    grid_search = GridSearchCV(
        best_model, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1
    )
    
    grid_search.fit(X_train, y_train)
    
    print(f"\nBest Parameters: {grid_search.best_params_}")
    print(f"Best CV Score: {grid_search.best_score_:.4f}")
    
    # Evaluate tuned model
    tuned_model = grid_search.best_estimator_
    y_pred_tuned = tuned_model.predict(X_test)
    tuned_accuracy = accuracy_score(y_test, y_pred_tuned)
    
    print(f"\nTuned Model Test Accuracy: {tuned_accuracy:.4f}")
    print(f"Improvement: {(tuned_accuracy - best_accuracy):.4f}")
    
    # Use tuned model as final model
    if tuned_accuracy > best_accuracy:
        best_model = tuned_model
        print("\n✓ Using tuned model as final model")
    else:
        print("\n✓ Keeping original model (no improvement from tuning)")
else:
    print(f"No parameter grid defined for {best_model_name}")

## 8. Save Final Model

In [ ]:
# Save the best model
model_path = '../../data/processed/best_model.pkl'
joblib.dump(best_model, model_path)

print(f"\nFinal model saved to: {model_path}")
print(f"Model type: {best_model_name}")
print(f"Test Accuracy: {best_accuracy:.4f}")

# Save model metadata
model_metadata = {
    'model_name': best_model_name,
    'test_accuracy': best_accuracy,
    'features_used': numerical_features + categorical_features,
    'target_variable': target
}

import json
with open('../../data/processed/model_metadata.json', 'w') as f:
    json.dump(model_metadata, f, indent=4)

print("\nModel metadata saved successfully!")

## 9. Model Training Summary

In [ ]:
print("\n" + "="*100)
print(" " * 35 + "MODEL TRAINING SUMMARY")
print("="*100)

print(f"\n1. MODELS EVALUATED")
print("-" * 100)
for i, model_name in enumerate(results.keys(), 1):
    print(f"   {i}. {model_name}")

print(f"\n2. BEST MODEL")
print("-" * 100)
print(f"   • Model: {best_model_name}")
print(f"   • Test Accuracy: {best_accuracy:.4f}")
print(f"   • Cross-Validation Score: {results[best_model_name]['CV_Mean']:.4f} (+/- {results[best_model_name]['CV_Std']:.4f})")

print(f"\n3. TOP 3 IMPORTANT FACTORS")
print("-" * 100)
if hasattr(best_model.named_steps['classifier'], 'feature_importances_'):
    for i, (feat, imp) in enumerate(feature_importance_df.head(3).values, 1):
        print(f"   {i}. {feat}: {imp:.4f}")
else:
    print("   Feature importance not available for this model type")

print(f"\n4. MODEL USAGE")
print("-" * 100)
print(f"   • Model saved at: {model_path}")
print(f"   • Can be used to predict employee performance for hiring decisions")
print(f"   • Input: Employee attributes (features used in training)")
print(f"   • Output: Predicted performance rating")

print("\n" + "="*100)
print("Model training completed successfully!")
print("="*100)